In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
device= ("cuda" if torch.cuda.is_available else "cpu")
device

'cuda'

In [7]:
!pip install -q kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_data
!ls plantvillage_data

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [01:42<00:00, 21.4MB/s]

'plantvillage dataset'


In [9]:
!ls plantvillage_data

'plantvillage dataset'


In [10]:
!ls "plantvillage_data/plantvillage dataset"

color  grayscale  segmented


In [11]:
from torchvision import transforms
train_transforms= transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

In [12]:
!pip install split-folders

In [16]:
import splitfolders

splitfolders.ratio(
    "plantvillage_data/plantvillage dataset/color",
    output="plantvillage_split",
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

Copying files: 54305 files [00:11, 4887.39 files/s]


In [19]:
from torchvision.datasets import ImageFolder
train_dataset= ImageFolder(root='plantvillage_split/train', transform= train_transforms)
val_dataset= ImageFolder(root='plantvillage_split/val', transform= val_transforms)
test_dataset= ImageFolder(root='plantvillage_split/test', transform= val_transforms)

In [20]:
print(len(train_dataset), len(val_dataset), len(test_dataset))
print(train_dataset.classes)
print(train_dataset.class_to_idx)

37997 8129 8179
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'To

In [21]:
from torch.utils.data import DataLoader
train_loader= DataLoader(train_dataset, batch_size=32, shuffle=True,num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [23]:
images, labels= next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [24]:
from torchvision import models
model = models.resnet18(weights='IMAGENET1K_V1')
print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [26]:
num_classes= 38
model.fc= nn.Linear(model.fc.in_features, num_classes)

In [27]:
model= model.to(device)

In [28]:
print(model.fc)

Linear(in_features=512, out_features=38, bias=True)


In [29]:
for param in model.parameters():
  param.requires_grad= False
for param in model.fc.parameters():
  param.requires_grad= True

In [30]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

#This should print only fc.weight and fc.bias — nothing else.

fc.weight
fc.bias


In [35]:
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
"""
Loss function — CrossEntropyLoss:
This is the standard choice for multi-class classification (which is exactly what we have — 38 classes). It compares the model's predicted probabilities against the true label and outputs a single number representing "how wrong" the prediction was. Lower loss = better predictions. Internally, it combines a softmax (turns raw outputs into probabilities) + negative log-likelihood, so you don't need to apply softmax yourself in the model.
Optimizer — Adam:
This is the algorithm that actually updates the weights to reduce the loss, using gradients computed during backpropagation. Adam is a good default — it adapts the learning rate per-parameter and generally converges faster/more reliably than plain SGD, especially for beginners.
"""

In [ ]:
"""
Forward pass — feed images through the model, get predictions
Calculate loss — compare predictions to true labels
Backward pass — compute gradients (how much each weight contributed to the error)
Update weights — optimizer nudges weights to reduce error
"""

In [36]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [ ]:
"""
model.eval() — switches batchnorm/dropout to inference behavior (opposite of model.train())
torch.no_grad() — tells PyTorch not to track gradients here. We're not updating weights during evaluation, so this saves memory and speeds things up
torch.max(outputs, 1) — for each image, the model outputs 38 raw scores (one per class). torch.max(..., 1) finds the index of the highest score along dimension 1 (the class dimension) — that index is the predicted class
We just count how many predictions matched the true label, out of the total, and convert to a percentage
"""

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Val Acc: {val_acc:.2f}%")



Epoch 1/5, Loss: 0.2462, Val Acc: 94.93%
Epoch 2/5, Loss: 0.2258, Val Acc: 94.71%
Epoch 3/5, Loss: 0.2114, Val Acc: 94.56%
Epoch 4/5, Loss: 0.1958, Val Acc: 95.20%
Epoch 5/5, Loss: 0.1768, Val Acc: 95.47%


In [ ]:
test_acc = evaluate(model, test_loader)
print(f"Final Test Accuracy: {test_acc:.2f}%")



Final Test Accuracy: 95.51%


In [ ]:
"""
model.train() — puts the model in "training mode" (matters for certain layers like dropout/batchnorm, which behave differently during training vs. evaluation — ResNet uses batchnorm, so this matters) images, labels = images.to(device) — moves this batch's tensors onto the GPU, matching where the model lives optimizer.zero_grad() — critical step: PyTorch accumulates gradients by default, so we must clear old gradients before computing new ones each batch, or they'd add up incorrectly across batches outputs = model(images) — the forward pass; gives raw prediction scores (logits) for each of the 38 classes, per image loss.backward() — computes gradients for every trainable parameter via backpropagation optimizer.step() — actually updates the weights using those gradients running_loss — just for tracking; we accumulate loss per batch and average it at the end of the epoch to see if the model's improving
"""

In [40]:
torch.save(model.state_dict(), 'plant_disease_model.pth')

In [ ]:
"""
What state_dict() actually is: a Python dictionary mapping each layer's name to its learned weight tensors (e.g., {'conv1.weight': tensor(...), 'fc.weight': tensor(...), ...}). This is the standard, recommended way to save PyTorch models — it saves just the numbers the model learned, not the class definition itself. That means when you reload it later, you need to reconstruct the same architecture first, then load the weights into it.
"""

In [41]:
import json
with open('class_to_idx.json', 'w') as f:
  json.dump(train_dataset.class_to_idx, f)

In [42]:
from google.colab import files
files.download('plant_disease_model.pth')
files.download('class_to_idx.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

prediction pipeline


In [43]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load('plant_disease_model.pth', map_location=device))
model = model.to(device)
model.eval()

#Remember — state_dict() only saved the numbers, not the architecture. So
#we first recreate the exact
#same model structure, then load the weights into it:

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [44]:
from PIL import Image

image = Image.open('/content/plantvillage_split/test/Apple___Black_rot/02186b4f-a9e1-4d19-ae3d-6cfb0f4b106a___JR_FrgE.S 2828.JPG').convert('RGB')
image_tensor = val_transforms(image)
image_tensor = image_tensor.unsqueeze(0)
image_tensor = image_tensor.to(device)

In [ ]:
"""
unsqueeze(0) — this is important. Your model was trained expecting batches of images, shape [batch_size, 3, 224, 224]. A single image after transforms is shape [3, 224, 224] — missing the batch dimension. unsqueeze(0) adds a dimension of size 1 at the front, making it [1, 3, 224, 224] — a "batch" of one image.
"""

In [45]:
with torch.no_grad():
    output = model(image_tensor)
    probabilities = torch.softmax(output, dim=1)
    confidence, predicted_idx = torch.max(probabilities, 1)

predicted_class = train_dataset.classes[predicted_idx.item()]
print(f"Prediction: {predicted_class}, Confidence: {confidence.item()*100:.2f}%")

Prediction: Apple___Black_rot, Confidence: 100.00%
